In [27]:
# Setup: connect to v.db (read-only), load the FULL set of "scope" and
# "unconvertable" errors (1,347,409 + 153,444 rows) -- this is the complete
# population for both categories, not a sample.
import sqlite3
import pandas as pd
import re

DB_PATH = "/Users/anil/code/reroll-data/data/v.db"
con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
df = pd.read_sql_query(
    """
    SELECT project, filename, reroll_error
    FROM repodata_conversion
    WHERE reroll_error LIKE 'scope:%' OR reroll_error LIKE 'unconvertable:%'
    """,
    con,
)
len(df), df.head()

(1500853,
     project                           filename  \
 0  0fdb5604  0fdb5604-1.0.0a1-py3-none-any.whl   
 1  0fdb5604  0fdb5604-1.0.0a2-py3-none-any.whl   
 2  0fdb5604    0fdb5604-1.0.1-py3-none-any.whl   
 3  0fdb5604   0fdb5604-1.0.10-py3-none-any.whl   
 4  0fdb5604   0fdb5604-1.0.11-py3-none-any.whl   
 
                                                   reroll_error  
 0  unconvertable: UnresolvedCondaNameError: no mapper resol...  
 1  unconvertable: UnresolvedCondaNameError: no mapper resol...  
 2  unconvertable: UnresolvedCondaNameError: no mapper resol...  
 3  unconvertable: UnresolvedCondaNameError: no mapper resol...  
 4  unconvertable: UnresolvedCondaNameError: no mapper resol...  )

In [34]:
# Split the full population into "scope" and "unconvertable" error
# dataframes (cheap boolean-mask filters, no re-query).
scope_df = df[df["reroll_error"].str.startswith("scope:")].copy()
unconvertable_df = df[df["reroll_error"].str.startswith("unconvertable:")].copy()
len(scope_df), len(unconvertable_df)

(1347409, 153444)

In [37]:
# What exception types make up "scope" errors (full population)?
exc_type = scope_df["reroll_error"].str.extract(r"^scope:\s*(\w+)")
scope_df["exc_type"] = exc_type[0]
scope_df["exc_type"].value_counts()

exc_type
UnsupportedPlatformError              1163751
UnsupportedInterpreterError            178362
UnsupportedInterpreterVersionError       5296
Name: count, dtype: int64

In [38]:
# Flag filenames that look like pypy or python-2.x wheels, based on the wheel
# filename tag convention: {name}-{version}-{python tag}-{abi tag}-{platform}.whl
# python tag examples: pp27/pp36 (pypy), cp27 (cpython 2.7), py2, py27
pypy_or_py2 = scope_df["filename"].str.contains(r"-(?:pp\d+|cp2\d|py2\d?)-", regex=True)
scope_df["pypy_or_py2"] = pypy_or_py2

summary = scope_df.groupby("exc_type")["pypy_or_py2"].agg(["sum", "count", "mean"])
summary["mean"] = (summary["mean"] * 100).round(1)
summary.columns = ["pypy_or_py2_count", "n", "pct_pypy_or_py2"]
summary["pct_of_all_scope"] = (summary["n"] / len(scope_df) * 100).round(1)
print(
    f"Overall: {pypy_or_py2.sum():,} / {len(scope_df):,} "
    f"({pypy_or_py2.mean() * 100:.1f}%) scope-error filenames look like pypy/py2"
)
summary.sort_values("n", ascending=False)

Overall: 178,271 / 1,347,409 (13.2%) scope-error filenames look like pypy/py2


,pypy_or_py2_count,n,pct_pypy_or_py2,pct_of_all_scope
exc_type,,,,
UnsupportedPlatformError,0,1163751,0.0,86.4
UnsupportedInterpreterError,178271,178362,99.9,13.2
UnsupportedInterpreterVersionError,0,5296,0.0,0.4


In [39]:
# Dig into UnsupportedPlatformError (39.5% of scope errors, 0% pypy/py2 --
# --allow-pre can't touch this bucket at all). Extract the platform tag from
# the wheel filename (second-to-last dash-separated field, minus .whl) to see
# what's actually driving these rejections.
platform_errs = scope_df[scope_df["exc_type"] == "UnsupportedPlatformError"].copy()


def extract_platform_tag(filename: str) -> str:
    # {name}-{version}[-{build}]-{python tag}-{abi tag}-{platform tag}.whl
    stem = filename[: -len(".whl")]
    return stem.rsplit("-", 1)[-1]


platform_errs["platform_tag"] = platform_errs["filename"].apply(extract_platform_tag)
print(f"{len(platform_errs):,} UnsupportedPlatformError rows")
platform_errs["platform_tag"].value_counts().head(30)

1,163,751 UnsupportedPlatformError rows


platform_tag
win32                                                                        228504
musllinux_1_2_x86_64                                                         157243
musllinux_1_2_aarch64                                                        110152
musllinux_1_2_i686                                                            63780
musllinux_1_1_x86_64                                                          63762
manylinux_2_17_ppc64le.manylinux2014_ppc64le                                  58391
manylinux_2_17_armv7l.manylinux2014_armv7l                                    55822
manylinux_2_17_i686.manylinux2014_i686                                        55813
manylinux_2_17_s390x.manylinux2014_s390x                                      53569
musllinux_1_2_armv7l                                                          39240
musllinux_1_1_i686                                                            32900
manylinux_2_5_i686.manylinux1_i686                             

In [40]:
# Bucket every platform tag into a human reason, using reroll's documented
# exclusions (docs/wheel_filename.md): musllinux, 32-bit OSes, iOS/Android,
# non-manylinux-glibc linux, non-x86_64/arm64/universal2 mac, and unsupported
# (non-x86_64/aarch64) architectures on otherwise-valid manylinux tags.


def bucket_platform(tag: str) -> str:
    if "musllinux" in tag:
        return "musllinux (musl libc)"
    if tag == "win32" or tag == "win_ia64":
        return "32-bit / non-amd64-arm64 windows"
    if "ios_" in tag or "iphoneos" in tag:
        return "iOS"
    if "android" in tag:
        return "Android"
    if "emscripten" in tag or "wasi" in tag or "wasm" in tag:
        return "wasm (emscripten/wasi)"
    if tag.startswith("macosx"):
        if re.search(r"(x86_64|arm64|universal2)$", tag):
            return "macOS (supported arch, other reason)"
        return "macOS unsupported arch (intel/ppc/fat/i386)"
    if tag.startswith("win"):
        return "windows other"
    if tag.startswith("manylinux") or tag.startswith("linux"):
        if re.search(r"(i686|i386)", tag):
            return "linux 32-bit (i686)"
        if re.search(r"(ppc64le|ppc64|s390x|armv7l|riscv64|loongarch64|mips)", tag):
            return "linux unsupported arch (ppc64le/s390x/armv7l/etc.)"
        if tag.startswith("linux_"):
            return "bare linux_* (non-manylinux glibc tag)"
        return "linux other"
    return "other/unrecognized"


platform_errs["reason"] = platform_errs["platform_tag"].apply(bucket_platform)
reason_counts = platform_errs["reason"].value_counts()
reason_pct = (reason_counts / len(platform_errs) * 100).round(1)
pd.DataFrame({"count": reason_counts, "pct_of_platform_errors": reason_pct})

,count,pct_of_platform_errors
reason,,
musllinux (musl libc),517924,44.5
32-bit / non-amd64-arm64 windows,228535,19.6
linux unsupported arch (ppc64le/s390x/armv7l/etc.),209385,18.0
linux 32-bit (i686),190009,16.3
macOS unsupported arch (intel/ppc/fat/i386),13666,1.2
bare linux_* (non-manylinux glibc tag),1684,0.1
Android,1517,0.1
iOS,573,0.0
wasm (emscripten/wasi),450,0.0


In [41]:
pd.set_option("display.max_colwidth", 60)
summary_platform = pd.DataFrame(
    {"count": reason_counts, "pct_of_platform_errors": reason_pct}
)
for reason, row in summary_platform.iterrows():
    print(f"{row['count']:>9,.0f}  {row['pct_of_platform_errors']:>5.1f}%  {reason}")

  517,924   44.5%  musllinux (musl libc)
  228,535   19.6%  32-bit / non-amd64-arm64 windows
  209,385   18.0%  linux unsupported arch (ppc64le/s390x/armv7l/etc.)
  190,009   16.3%  linux 32-bit (i686)
   13,666    1.2%  macOS unsupported arch (intel/ppc/fat/i386)
    1,684    0.1%  bare linux_* (non-manylinux glibc tag)
    1,517    0.1%  Android
      573    0.0%  iOS
      450    0.0%  wasm (emscripten/wasi)
        8    0.0%  windows other


In [42]:
# How many distinct packages actually account for the 463,205 musllinux rows?
musl = platform_errs[platform_errs["reason"] == "musllinux (musl libc)"]
n_projects = musl["project"].nunique()
print(f"{len(musl):,} musllinux wheel rows from {n_projects:,} distinct projects")
print(f"-> average {len(musl) / n_projects:.1f} musllinux wheels per project")

top_projects = musl["project"].value_counts()
top_projects.head(20)

517,924 musllinux wheel rows from 6,195 distinct projects
-> average 83.6 musllinux wheels per project


project
ddtrace                 11483
aioesphomeapi            4038
stringzilla              3417
aiohttp                  2808
dbus-fast                2663
simsimd                  2631
clickhouse-connect       2054
zeroconf                 1981
pydantic_core            1923
habluetooth              1906
coverage                 1888
bitarray                 1744
RapidFuzz                1741
passagemath-coxeter3     1646
ruff                     1644
attoworld                1608
passagemath-homfly       1598
yarl                     1525
grpcio                   1475
grpcio-tools             1475
Name: count, dtype: int64

In [43]:
# Why 77 musllinux wheels/project on average? Check the shape of the
# distribution, and how much of it is just multiple releases x multiple
# python versions x multiple musl tags (1_1 vs 1_2) x multiple arches for
# the same handful of popular C-extension packages.
print("Distribution of musllinux-row-count per project:")
print(top_projects.describe())
print()
print("Cumulative share of rows held by the top N projects:")
cum_share = top_projects.cumsum() / len(musl) * 100
for n in [10, 50, 100, 500, 1000, 6000]:
    print(
        f"  top {n:>5,} projects -> {cum_share.iloc[min(n, len(cum_share)) - 1]:.1f}% of musllinux rows"
    )

# Sanity check the "many wheels per release" theory for the #1 project
ddtrace = musl[musl["project"] == "ddtrace"]
print(
    f"\nddtrace: {len(ddtrace):,} musllinux rows, "
    f"{ddtrace['filename'].str.extract(r'-(\d+\.\d+\.\d+)')[0].nunique()} distinct versions seen in filenames"
)

Distribution of musllinux-row-count per project:
count     6195.000000
mean        83.603551
std        232.924461
min          1.000000
25%         10.000000
50%         28.000000
75%         76.000000
max      11483.000000
Name: count, dtype: float64

Cumulative share of rows held by the top N projects:
  top    10 projects -> 6.7% of musllinux rows
  top    50 projects -> 16.6% of musllinux rows
  top   100 projects -> 24.4% of musllinux rows
  top   500 projects -> 52.5% of musllinux rows
  top 1,000 projects -> 68.7% of musllinux rows
  top 6,000 projects -> 99.9% of musllinux rows

ddtrace: 11,483 musllinux rows, 493 distinct versions seen in filenames


In [44]:
# --- New section: unconvertable ("unparseable") errors, full population ---
# unconvertable_df already loaded in memory (153,444 rows). Inspect the raw
# reroll_error string format before deciding how to bucket it into sub-types.
print(len(unconvertable_df))
for s in unconvertable_df["reroll_error"].head(10):
    print(repr(s))

153444
"unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'modal': candidates=(Candidate(conda_name='modal',
 probability=0.3325, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'), Candidate(conda_name=
'modal-client', probability=0.57, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'))"
"unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'modal': candidates=(Candidate(conda_name='modal',
 probability=0.3325, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'), Candidate(conda_name=
'modal-client', probability=0.57, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'))"
"unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'modal': candidates=(Candidate(conda_name='modal',
 probability=0.3325, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'),

In [45]:
# What exception types make up "unconvertable" errors (full population)?
exc_type = unconvertable_df["reroll_error"].str.extract(r"^unconvertable:\s*(\w+)")
unconvertable_df["exc_type"] = exc_type[0]
counts = unconvertable_df["exc_type"].value_counts()
pct = (counts / len(unconvertable_df) * 100).round(1)
pd.DataFrame({"count": counts, "pct_of_unconvertable": pct})

,count,pct_of_unconvertable
exc_type,,
UnresolvedCondaNameError,86859,56.6
PythonRangeMismatchError,35549,23.2
UnconvertableMarkerError,21394,13.9
UnconvertableRequirementError,8365,5.5
InvalidCondaNameError,1277,0.8


In [46]:
# For each unconvertable exc_type: how many distinct projects (packages being
# rerolled) and distinct wheel filenames does it touch? Distinguishes "one huge
# project is spamming this bucket" from "broad-based, many projects affected".
by_type = unconvertable_df.groupby("exc_type").agg(
    n_rows=("reroll_error", "size"),
    n_projects=("project", "nunique"),
    n_filenames=("filename", "nunique"),
)
by_type["rows_per_project"] = (by_type["n_rows"] / by_type["n_projects"]).round(1)
by_type.sort_values("n_rows", ascending=False)

,n_rows,n_projects,n_filenames,rows_per_project
exc_type,,,,
UnresolvedCondaNameError,86859,4243,86841,20.5
PythonRangeMismatchError,35549,1437,35545,24.7
UnconvertableMarkerError,21394,1533,21394,14.0
UnconvertableRequirementError,8365,617,8365,13.6
InvalidCondaNameError,1277,114,1277,11.2


In [47]:
# Look at one representative message per exc_type to understand each format
# before writing extraction regexes for sub-types.
for t, grp in unconvertable_df.groupby("exc_type"):
    print("=" * 80)
    print(t, f"(n={len(grp)})")
    print(grp["reroll_error"].iloc[0])
    print()

InvalidCondaNameError (n=1277)
unconvertable: InvalidCondaNameError: conda package name 'iptv-subscription-ip-tv-subscription-iptv-premium-subscription-worldwi
de-channels' exceeds 64 characters (81)

PythonRangeMismatchError (n=35549)
unconvertable: PythonRangeMismatchError: bmctoolkit: filename-implied python range python >=3.9,<3.10.0a0 does not intersect Req
uires-Python '>=3.9.6'; no depends generated

UnconvertableMarkerError (n=21394)
unconvertable: UnconvertableMarkerError: cannot convert the marker in 'typing-extensions>=4.10.0; python_version < "3.11.0"' to
a matchspec: python_version literal '3.11.0' is not a major.minor version

UnconvertableRequirementError (n=8365)
unconvertable: UnconvertableRequirementError: cannot convert 'aleksis-app-resint==3.0.dev0+20220802190124.c3757b88': it has a lo
cal version label

UnresolvedCondaNameError (n=86859)
unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'modal': candidates=(Candidate(conda_name='modal',
p

In [48]:
# UnconvertableMarkerError (13.9%, 21,394 rows): extract the trailing "reason"
# clause after "to a matchspec: ", then strip quoted literals/version numbers
# so we can cluster by message *shape* rather than exact text.
marker_errs = unconvertable_df[
    unconvertable_df["exc_type"] == "UnconvertableMarkerError"
].copy()
reason = marker_errs["reroll_error"].str.extract(r"to a matchspec:\s*(.+)$")[0]
marker_errs["reason"] = reason
# Generalize: drop quoted strings and bare version-like tokens to get a template
template = reason.str.replace(r"'[^']*'", "<LIT>", regex=True).str.replace(
    r"\b\d+(?:\.\d+)*\b", "<NUM>", regex=True
)
marker_errs["reason_template"] = template
counts = marker_errs["reason_template"].value_counts()
pd.DataFrame({"count": counts, "pct": (counts / len(marker_errs) * 100).round(1)})

,count,pct
reason_template,,
python_version literal <LIT> is not a major.minor version,18807,87.9
"<LIT>/<LIT> markers are not supported for matchspec conversion: python_version not in ""<NUM>, <NUM>, <NUM>, <NUM>""",1053,4.9
comparator <LIT> is not supported for python_version,438,2.0
"<LIT>/<LIT> markers are not supported for matchspec conversion: python_full_version not in ""<NUM>, <NUM>""",196,0.9
"<LIT>/<LIT> markers are not supported for matchspec conversion: python_version not in ""<NUM>, <NUM>, <NUM>""",125,0.6
"<LIT>/<LIT> markers are not supported for matchspec conversion: python_version not in ""<NUM>, <NUM>, <NUM>, <NUM>, <NUM>, <NUM>, <NUM>""",61,0.3
"<LIT>/<LIT> markers are not supported for matchspec conversion: python_version not in ""<NUM>, <NUM>, <NUM>, <NUM>, <NUM>""",61,0.3
"<LIT>/<LIT> markers are not supported for matchspec conversion: python_version not in ""<NUM>""",43,0.2
"<LIT>/<LIT> markers are not supported for matchspec conversion: python_version in ""<NUM> <NUM> <NUM>""",34,0.2


In [49]:
# Un-truncate the "<LIT>/<LIT> markers are not supported..." family, and check
# what the actual literal python_version values are in the dominant bucket.
pd.set_option("display.max_colwidth", 200)
other_markers = marker_errs.loc[
    marker_errs["reason_template"].str.startswith("<LIT>/<LIT> markers"), "reason"
]
print("distinct 'X/Y markers not supported' variants:")
print(other_markers.value_counts())
print()
top_bucket = marker_errs.loc[
    marker_errs["reason_template"]
    == "python_version literal <LIT> is not a major.minor version",
    "reason",
]
print("sample literal python_version values (top bucket):")
print(top_bucket.str.extract(r"literal '([^']*)'")[0].value_counts().head(15))

distinct 'X/Y markers not supported' variants:
reason
'in'/'not in' markers are not supported for matchspec conversion: python_version not in "3.0, 3.1, 3.2, 3.3"
         1053
'in'/'not in' markers are not supported for matchspec conversion: python_full_version not in "3.9.0, 3.9.1"
          195
'in'/'not in' markers are not supported for matchspec conversion: python_version not in "3.0, 3.1, 3.2"
          125
'in'/'not in' markers are not supported for matchspec conversion: python_version not in "3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6"
           61
'in'/'not in' markers are not supported for matchspec conversion: python_version not in "3.0, 3.1, 3.2, 3.3, 3.4"
           61
'in'/'not in' markers are not supported for matchspec conversion: python_version not in "3.13"
           43
'in'/'not in' markers are not supported for matchspec conversion: python_version in "3.3 3.4 3.5"
           28
'in'/'not in' markers are not supported for matchspec conversion: python_version not in "3.0, 3.

In [50]:
# PythonRangeMismatchError (23.2%, 35,549 rows): "filename-implied python range
# X does not intersect Requires-Python 'Y'". Extract both ranges to see whether
# this is mostly a narrow micro-version technicality (e.g. >=3.9 vs >=3.9.6,
# which DO overlap in reality) or genuinely disjoint ranges.
range_errs = unconvertable_df[
    unconvertable_df["exc_type"] == "PythonRangeMismatchError"
].copy()
extracted = range_errs["reroll_error"].str.extract(
    r"filename-implied python range (?P<filename_range>[^\s]+(?:\s*[^\s]+)*?) does not intersect Requires-Python '(?P<requires_python>[^']*)'"
)
range_errs["filename_range"] = extracted["filename_range"]
range_errs["requires_python"] = extracted["requires_python"]
print(range_errs[["filename_range", "requires_python"]].isna().sum())
range_errs[["filename_range", "requires_python"]].value_counts().head(20)

filename_range     0
requires_python    0
dtype: int64


filename_range           requires_python
python >=3.9,<3.10.0a0   >=3.10             4725
python >=3.8,<3.9.0a0    >=3.9              3314
                         >=3.10             3093
python >=3.7,<3.8.0a0    >=3.8              2518
python >=3.10,<3.11.0a0  >=3.11             1960
python >=3.6,<3.7.0a0    >=3.7              1644
                         >=3.8              1611
python >=3.14,<3.15.0a0  <3.14,>=3.10       1098
                         <3.14,>=3.9         960
python >=3.12,<3.13.0a0  <4.0,>=3.13         946
python >=3.9,<3.10.0a0   >=3.9.16            812
python >=3.10,<3.11.0a0  >=3.12              679
python >=3.13,<3.14.0a0  <3.13,>=3.10        504
python >=3.7,<3.8.0a0    >=3.9               503
                         >=3.10              467
python >=3.10,<3.11.0a0  <3.14,>=3.11        409
python >=3.13,<3.14.0a0  <3.13,>=3.8         375
python >=3.10,<3.11.0a0  >=3.10.1            375
                         >=3.13              372
python >=3.14,<3.15.0a0  <3.

In [51]:
# Some pairs above (e.g. "python >=3.9,<3.10.0a0" vs requires_python ">=3.9.16")
# look like they SHOULD intersect (3.9.16..3.9.99 satisfies both). Check with
# `packaging` whether reroll's "does not intersect" call is a real disjoint
# range or a false positive, on the *unique* (filename_range, requires_python)
# pairs only (cheap - a few hundred/thousand uniques, not 35k rows).
from packaging.specifiers import SpecifierSet
from packaging.version import Version

pair_counts = (
    range_errs.groupby(["filename_range", "requires_python"])
    .size()
    .rename("n")
    .reset_index()
)
print(
    f"{len(pair_counts)} unique (filename_range, requires_python) pairs covering {pair_counts['n'].sum()} rows"
)

candidates = [
    Version(f"{maj}.{minr}.{mic}")
    for maj in range(0, 5)
    for minr in range(0, 30)
    for mic in (0, 1, 2, 5, 9, 15, 16, 20, 50, 99)
]


def truly_intersects(filename_range, requires_python):
    try:
        fs = SpecifierSet(filename_range.replace("python ", "").strip())
        rs = SpecifierSet(requires_python)
    except Exception:
        return None
    return any(
        fs.contains(v, prereleases=True) and rs.contains(v, prereleases=True)
        for v in candidates
    )


pair_counts["truly_intersects"] = pair_counts.apply(
    lambda r: truly_intersects(r["filename_range"], r["requires_python"]), axis=1
)
pair_counts.groupby("truly_intersects")["n"].agg(["size", "sum"])

305 unique (filename_range, requires_python) pairs covering 35549 rows


,size,sum
truly_intersects,,
False,187,29651
True,118,5898


In [52]:
print(
    "Pairs flagged 'mismatch' by reroll but ACTUALLY intersect (likely a reroll bug),"
)
print("sorted by row count:")
display(
    pair_counts[pair_counts["truly_intersects"] == True]
    .sort_values("n", ascending=False)
    .head(15)
)

print()
print("Pairs that are GENUINELY disjoint (real metadata incompatibility),")
print("sorted by row count:")
display(
    pair_counts[pair_counts["truly_intersects"] == False]
    .sort_values("n", ascending=False)
    .head(15)
)

Pairs flagged 'mismatch' by reroll but ACTUALLY intersect (likely a reroll bug),
sorted by row count:



Pairs that are GENUINELY disjoint (real metadata incompatibility),
sorted by row count:


,filename_range,requires_python,n,truly_intersects
297,"python >=3.9,<3.10.0a0",>=3.9.16,812,True
25,"python >=3.10,<3.11.0a0",>=3.10.1,375,True
288,"python >=3.9,<3.10.0a0",>3.9,297,True
156,"python >=3.6,<3.7.0a0",>=3.6.1,286,True
193,"python >=3.7,<3.8.0a0","<4.0.0,>=3.7.1",277,True
135,"python >=3.5,<3.6.0a0",>=3.5.3,272,True
295,"python >=3.9,<3.10.0a0",>=3.9.13,213,True
244,"python >=3.8,<3.9.0a0","<4.0.0,>=3.8.1",195,True
204,"python >=3.7,<3.8.0a0",>=3.7.1,185,True
296,"python >=3.9,<3.10.0a0",>=3.9.14,175,True


,filename_range,requires_python,n,truly_intersects
290,"python >=3.9,<3.10.0a0",>=3.10,4725,False
264,"python >=3.8,<3.9.0a0",>=3.9,3314,False
252,"python >=3.8,<3.9.0a0",>=3.10,3093,False
214,"python >=3.7,<3.8.0a0",>=3.8,2518,False
30,"python >=3.10,<3.11.0a0",>=3.11,1960,False
163,"python >=3.6,<3.7.0a0",>=3.7,1644,False
166,"python >=3.6,<3.7.0a0",>=3.8,1611,False
105,"python >=3.14,<3.15.0a0","<3.14,>=3.10",1098,False
112,"python >=3.14,<3.15.0a0","<3.14,>=3.9",960,False
74,"python >=3.12,<3.13.0a0","<4.0,>=3.13",946,False


In [53]:
# UnconvertableRequirementError (5.5%, 8,365 rows): "cannot convert '<req>': <reason>"
req_errs = unconvertable_df[
    unconvertable_df["exc_type"] == "UnconvertableRequirementError"
].copy()
reason = req_errs["reroll_error"].str.extract(r"cannot convert '[^']*':\s*(.+)$")[0]
req_errs["reason"] = reason
template = reason.str.replace(r"'[^']*'", "<LIT>", regex=True)
req_errs["reason_template"] = template
counts = req_errs["reason_template"].value_counts()
pd.DataFrame({"count": counts, "pct": (counts / len(req_errs) * 100).round(1)})

,count,pct
reason_template,,
it has a local version label,5368,64.2
it has a direct URL reference,93,1.1


In [54]:
# 8,365 total but only 5,461 matched above -- check the unmatched rows' raw text.
unmatched = req_errs[req_errs["reason"].isna()]
print(len(unmatched))
for s in unmatched["reroll_error"].head(5):
    print(repr(s))
    print()

2904
"unconvertable: UnconvertableRequirementError: 'cloud files deployment' is not a legal conda extra name (CEP-29): must be 1-64 c
haracters of [a-z0-9_.+-]"

"unconvertable: UnconvertableRequirementError: 'cloud files deployment' is not a legal conda extra name (CEP-29): must be 1-64 c
haracters of [a-z0-9_.+-]"

"unconvertable: UnconvertableRequirementError: 'cloud files deployment' is not a legal conda extra name (CEP-29): must be 1-64 c
haracters of [a-z0-9_.+-]"

"unconvertable: UnconvertableRequirementError: 'cloud files deployment' is not a legal conda extra name (CEP-29): must be 1-64 c
haracters of [a-z0-9_.+-]"

"unconvertable: UnconvertableRequirementError: 'gcs deployment' is not a legal conda extra name (CEP-29): must be 1-64 character
s of [a-z0-9_.+-]"


In [55]:
# Redo with both message shapes covered:
#   "cannot convert '<req>': <reason>"
#   "'<extra>' is not a legal conda extra name (CEP-29): <reason>"
def classify_req_error(msg):
    m = re.search(r"cannot convert '[^']*':\s*(.+)$", msg)
    if m:
        return m.group(1)
    m = re.search(r"is not a legal conda extra name \(CEP-29\):\s*(.+)$", msg)
    if m:
        return f"illegal conda extra name: {m.group(1)}"
    return "OTHER/unrecognized"


req_errs["reason2"] = req_errs["reroll_error"].map(classify_req_error)
counts = req_errs["reason2"].value_counts()
pd.DataFrame({"count": counts, "pct": (counts / len(req_errs) * 100).round(1)})

,count,pct
reason2,,
it has a local version label,5368,64.2
OTHER/unrecognized,2232,26.7
illegal conda extra name: must be 1-64 characters of [a-z0-9_.+-],672,8.0
it has a direct URL reference,93,1.1


In [56]:
unrecognized = req_errs[req_errs["reason2"] == "OTHER/unrecognized"]
for s in unrecognized["reroll_error"].drop_duplicates().head(10):
    print(repr(s))
    print()

"unconvertable: UnconvertableRequirementError: 'amulet-compiler-version ==3.0.0.309027957683952396889703.15.0.0.15000309', conve
rted from 'amulet-compiler-version==3.0.0.309027957683952396889703.15.0.0.15000309', is not a valid matchspec"

"unconvertable: UnconvertableRequirementError: 'amulet-compiler-version ==4.309027957683952396889703.17', converted from 'amulet
-compiler-version==4.309027957683952396889703.17', is not a valid matchspec"

"unconvertable: UnconvertableRequirementError: 'amulet-compiler-version ==1.3.0.309027957683952396889703.15.0.0.15000309', conve
rted from 'amulet-compiler-version==1.3.0.309027957683952396889703.15.0.0.15000309', is not a valid matchspec"

"unconvertable: UnconvertableRequirementError: 'amulet-compiler-version ==3.0.0.309027957683952396889703.15.0.0.15000100', conve
rted from 'amulet-compiler-version==3.0.0.309027957683952396889703.15.0.0.15000100', is not a valid matchspec"

"unconvertable: UnconvertableRequirementError: 'amulet-compiler-versio

In [57]:
# That 3rd shape is "'<attempted-matchspec>', converted from '<orig>', is not a
# valid matchspec". Check how concentrated it is by dependency name, and how
# many distinct *rerolled projects* (not the offending dependency) it touches.
invalid_matchspec = unrecognized["reroll_error"].str.extract(
    r"^unconvertable: UnconvertableRequirementError: '([^\s']+)"
)[0]
print("Top offending dependency names inside 'is not a valid matchspec' errors:")
print(invalid_matchspec.value_counts().head(10))
print()
print(f"Total OTHER/unrecognized rows: {len(unrecognized)}")
print(f"Distinct rerolled projects affected: {unrecognized['project'].nunique()}")
amulet_share = (invalid_matchspec == "amulet-compiler-version").sum()
print(
    f"'amulet-compiler-version' dependency alone: {amulet_share} rows ({amulet_share / len(unrecognized) * 100:.1f}% of this sub-bucket)"
)

Top offending dependency names inside 'is not a valid matchspec' errors:
0
numpy                      720
amulet-compiler-version    416
zarr                       239
importlib-metadata         188
pytest                     146
astropy-base               126
importlib_resources         94
stable-retro-turbo          87
lxml                        40
orjson                      32
Name: count, dtype: int64

Total OTHER/unrecognized rows: 2232
Distinct rerolled projects affected: 62
'amulet-compiler-version' dependency alone: 416 rows (18.6% of this sub-bucket)


In [58]:
# UnresolvedCondaNameError (56.6%, 86,859 rows - the biggest bucket by far):
# "no mapper resolved a conda name for '<name>': candidates=(...)"
# Split into: zero candidates (genuinely unknown to any mapper) vs >=1
# candidate (found options but none crossed the confidence threshold).
unres = unconvertable_df[
    unconvertable_df["exc_type"] == "UnresolvedCondaNameError"
].copy()
dep_name = unres["reroll_error"].str.extract(
    r"no mapper resolved a conda name for '([^']*)'"
)[0]
unres["dep_name"] = dep_name
has_candidates = unres["reroll_error"].str.contains(
    r"candidates=\(Candidate", regex=True
)
unres["has_candidates"] = has_candidates
n_candidates = unres["reroll_error"].str.count(r"Candidate\(")
unres["n_candidates"] = n_candidates

print(unres["has_candidates"].value_counts())
print()
print("distribution of n_candidates among rows that DO have candidates:")
print(unres.loc[has_candidates, "n_candidates"].value_counts().sort_index())
print()
print(f"distinct unresolved dependency names: {unres['dep_name'].nunique()}")
print("top unresolved dependency names by row count:")
print(unres["dep_name"].value_counts().head(15))

has_candidates
True    86859
Name: count, dtype: int64

distribution of n_candidates among rows that DO have candidates:
n_candidates
1       21
2    86821
3       17
Name: count, dtype: int64

distinct unresolved dependency names: 18
top unresolved dependency names by row count:
dep_name
onnxruntime             58824
modal                   10758
onnxruntime-gpu          6074
pyqtwebengine            3812
dspy                     3741
scikit-learn-intelex     1847
functools32              1039
pyqtchart                 554
mdahole2                   55
mathicsscript              46
pylibiio                   42
alchemiscale               17
pyplaid                    17
simhash-py                 13
oasis                       7
Name: count, dtype: int64


In [59]:
# Quantify the onnxruntime/modal finding against the FULL unconvertable population,
# and check InvalidCondaNameError (0.8%, smallest bucket) for message shape variety.
total = len(unconvertable_df)
print("Top unresolved-name offenders as % of ALL unconvertable errors:")
top_names = unres["dep_name"].value_counts()
print((top_names / total * 100).round(1).head(8))
print(
    f"\nTop 3 names alone ({top_names.head(3).sum():,} rows) = "
    f"{top_names.head(3).sum() / total * 100:.1f}% of ALL 153,444 unconvertable errors"
)

print()
invalid_errs = unconvertable_df[unconvertable_df["exc_type"] == "InvalidCondaNameError"]
shape = invalid_errs["reroll_error"].str.extract(r"InvalidCondaNameError:\s*(.+?)\s*'")[
    0
]
print("InvalidCondaNameError message shapes:")
print(shape.value_counts())

Top unresolved-name offenders as % of ALL unconvertable errors:
dep_name
onnxruntime             38.3
modal                    7.0
onnxruntime-gpu          4.0
pyqtwebengine            2.5
dspy                     2.4
scikit-learn-intelex     1.2
functools32              0.7
pyqtchart                0.4
Name: count, dtype: float64

Top 3 names alone (75,656 rows) = 49.3% of ALL 153,444 unconvertable errors

InvalidCondaNameError message shapes:
0
conda package name    1275
'tааmo                   1
'tаааааmo                1
Name: count, dtype: int64


In [60]:
# InvalidCondaNameError sample of long names + the 2 homoglyph/spam outliers
long_names = (
    invalid_errs["reroll_error"]
    .str.extract(r"conda package name '([^']*)' exceeds")[0]
    .dropna()
)
print(
    f"{long_names.nunique()} distinct over-length names across {invalid_errs['project'].nunique()} projects"
)
print(
    "length distribution:",
    long_names.str.len().describe()[["min", "50%", "max"]].to_dict(),
)
print()
print("sample long names (SEO/keyword-stuffed PyPI project names):")
print(long_names.drop_duplicates().head(8).to_list())
print()
weird = invalid_errs[
    ~invalid_errs["reroll_error"].str.contains("exceeds 64 characters")
]
print("non-length InvalidCondaNameError rows:")
print(weird[["project", "reroll_error"]].to_string())

113 distinct over-length names across 114 projects
length distribution: {'min': 65.0, '50%': 68.0, 'max': 188.0}

sample long names (SEO/keyword-stuffed PyPI project names):
['iptv-subscription-ip-tv-subscription-iptv-premium-subscription-worldwide-channels', 'ks903naturalintonationaivoice-bate-vr-fin
allast-librarypackages-datas', 'lucy-1986-2009-project-ethereum-market-prediction-unit-limit-uncertainty-control-yourself-ourfri
endlucy', 'aws-solutions-constructs-aws-dynamodb-stream-lambda-elasticsearch-kibana', 'aws-solutions-constructs-aws-dynamodbstre
ams-lambda-elasticsearch-kibana', 'aws-solutions-constructs-aws-kinesis-firehose-s3-kinesis-analytics', 'aws-solutions-konstruk-
aws-dynamodb-stream-lambda-elasticsearch-kibana', 'cdk-cloudformation-awscommunity-applicationautoscaling-scheduledaction']

non-length InvalidCondaNameError rows:
        project                                                                                 reroll_error
1293227    t-mo     unconvertable: In

In [61]:
# Roll everything up: one table, sub-type counts as % of the FULL 153,444-row
# unconvertable population, for the write-up below.
rows = [
    ("UnresolvedCondaNameError", "onnxruntime unresolved", 58824),
    ("UnresolvedCondaNameError", "modal unresolved", 10758),
    ("UnresolvedCondaNameError", "onnxruntime-gpu unresolved", 6074),
    (
        "UnresolvedCondaNameError",
        "other 15 dep names unresolved",
        86859 - 58824 - 10758 - 6074,
    ),
    ("PythonRangeMismatchError", "genuine Requires-Python drift (disjoint)", 29651),
    ("PythonRangeMismatchError", "false positive - reroll intersection bug", 5898),
    ("UnconvertableMarkerError", "python_version literal not major.minor", 18807),
    (
        "UnconvertableMarkerError",
        "'in'/'not in' marker unsupported",
        21394 - 18807 - 438 - 5,
    ),
    (
        "UnconvertableMarkerError",
        "unsupported comparator for python_version(_full)",
        438 + 5,
    ),
    ("UnconvertableRequirementError", "PEP 440 local version label", 5368),
    (
        "UnconvertableRequirementError",
        "matchspec generation bug (invalid output)",
        2232,
    ),
    ("UnconvertableRequirementError", "illegal conda extra name (CEP-29)", 672),
    ("UnconvertableRequirementError", "direct URL reference", 93),
    ("InvalidCondaNameError", "name >64 chars (SEO-spam project names)", 1275),
    ("InvalidCondaNameError", "homoglyph/Cyrillic lookalike name", 2),
]
summary_all = pd.DataFrame(rows, columns=["exc_type", "sub_reason", "n_rows"])
summary_all["pct_of_all_unconvertable"] = (
    summary_all["n_rows"] / len(unconvertable_df) * 100
).round(1)
assert summary_all["n_rows"].sum() == len(unconvertable_df), summary_all["n_rows"].sum()
summary_all.sort_values("n_rows", ascending=False).reset_index(drop=True)

,exc_type,sub_reason,n_rows,pct_of_all_unconvertable
0,UnresolvedCondaNameError,onnxruntime unresolved,58824,38.3
1,PythonRangeMismatchError,genuine Requires-Python drift (disjoint),29651,19.3
2,UnconvertableMarkerError,python_version literal not major.minor,18807,12.3
3,UnresolvedCondaNameError,other 15 dep names unresolved,11203,7.3
4,UnresolvedCondaNameError,modal unresolved,10758,7.0
5,UnresolvedCondaNameError,onnxruntime-gpu unresolved,6074,4.0
6,PythonRangeMismatchError,false positive - reroll intersection bug,5898,3.8
7,UnconvertableRequirementError,PEP 440 local version label,5368,3.5
8,UnconvertableRequirementError,matchspec generation bug (invalid output),2232,1.5
9,UnconvertableMarkerError,'in'/'not in' marker unsupported,2144,1.4


## Unconvertable ("unparseable") errors: what causes the 153,444 rows

Unlike `scope` errors (out-of-scope wheels), `unconvertable` errors are wheels
reroll *tried* to convert but couldn't parse/represent as a conda package.
There are exactly 5 exception types, and just two of them (`UnresolvedCondaNameError`
+ `PythonRangeMismatchError`) account for **79.9%** of all 153,444 rows.

| exc_type | n | % of all unconvertable | top driver |
|---|---|---|---|
| `UnresolvedCondaNameError` | 86,859 | 56.6% | **onnxruntime alone = 38.3%** of everything |
| `PythonRangeMismatchError` | 35,549 | 23.2% | mostly genuine Requires-Python drift |
| `UnconvertableMarkerError` | 21,394 | 13.9% | non-major.minor `python_version` literals |
| `UnconvertableRequirementError` | 8,365 | 5.5% | PEP 440 local version labels |
| `InvalidCondaNameError` | 1,277 | 0.8% | SEO-spam PyPI project names >64 chars |

### 1. `UnresolvedCondaNameError` (56.6% of all errors) - the name mapper gave up
Only **18 distinct dependency names** produce every one of these 86,859 rows,
spread across 4,243 different rerolled projects (i.e. lots of packages depend
on a small number of unmapped names, not one project spamming the log).
The top 3 alone are **49.3% of the entire unconvertable population**:

- `onnxruntime` - 58,824 rows (**38.3% of ALL unconvertable errors**)
- `modal` - 10,758 rows (7.0%)
- `onnxruntime-gpu` - 6,074 rows (4.0%)

**Fixing the conda-name mapper for a handful of packages (starting with
`onnxruntime`) would eliminate roughly half of all unconvertable errors.**

### 2. `PythonRangeMismatchError` (23.2%) - filename python tag vs. Requires-Python
The wheel's filename-implied CPython range (e.g. `cp39` &rarr; `>=3.9,<3.10.0a0`)
doesn't intersect the sdist's declared `Requires-Python`. Splitting the 305
unique (filename_range, requires_python) pairs with `packaging`:

- **83.4% (29,651 rows) are genuinely disjoint** - real metadata drift where a
  later release raised the `Requires-Python` floor (e.g. `>=3.10`) but an
  older, narrower-tagged wheel (`cp39`) is still published under the same
  release stream.
- **16.6% (5,898 rows) are false positives caused by a bug in reroll's own
  range-intersection check**, e.g. filename implies `>=3.9,<3.10.0a0` and
  `Requires-Python` is `>=3.9.16` - these ranges *do* overlap (3.9.16-3.9.99),
  but reroll flags them as mismatched anyway. Worth filing as a bug; fixing it
  recovers ~5,900 wheels for conversion at no metadata cost.

### 3. `UnconvertableMarkerError` (13.9%) - marker can't become a matchspec
- **87.9% (18,807 rows):** `python_version` marker literal isn't major.minor
  granularity - either a bare major version (`"3"`, `"4"`) or a micro-pinned
  version (`"3.5.2"`, `"3.8.0"`). Conda matchspecs only support major.minor
  for `python_version`.
- **~10% (2,144 + 443 rows):** `in`/`not in` marker operators (e.g.
  `python_version not in "3.0, 3.1, 3.2, ..."`) and unsupported comparators -
  no direct matchspec equivalent exists for these marker forms.

### 4. `UnconvertableRequirementError` (5.5%)
- **64.2% (5,368 rows):** dependency pin has a PEP 440 local version label
  (e.g. `==3.0.dev0+20220802190124.c3757b88`) - not expressible in conda's
  version grammar.
- **26.7% (2,232 rows):** matchspec generation produced syntactically invalid
  output. ~19% of this slice is one pathological project
  (`amulet-compiler-version`, absurdly long version strings), but the rest
  is common packages (`numpy`, `zarr`, `importlib-metadata`, `pytest`, ...)
  tripped up by `python_full_version == "X.Y.*"` markers being rendered as
  invalid `[when=...]` matchspec selectors - likely another reroll bug.
- **8.0% (672 rows):** extra name fails CEP-29 (e.g. `"cloud files deployment"`
  has spaces, isn't normalized to `cloud-files-deployment`).
- **1.1% (93 rows):** direct URL reference in a requirement.

### 5. `InvalidCondaNameError` (0.8%) - smallest bucket
- **99.8% (1,275 rows):** conda name exceeds 64 characters - almost entirely
  SEO-spam/keyword-stuffed PyPI project names (114 distinct projects, e.g.
  `iptv-subscription-ip-tv-subscription-iptv-premium-subscription-...`).
- **2 rows:** homoglyph/Cyrillic lookalike characters in the project name
  (`t-mo` &rarr; `'tааmo'`), a typosquat-style spam package.

### Bottom line
- **~50% of all unconvertable errors** trace to just 3 unresolved dependency
  names, with `onnxruntime` alone responsible for over a third.
- **At least ~8,100 rows (~5.3%)** are reroll-side bugs, not real metadata
  problems: the range-intersection false positives (5,898) and the marker-to-
  `[when=...]` matchspec generation bug (2,232-ish). Fixing those two bugs is
  pure upside with zero mapper/metadata work required.
- The remaining bulk is genuine PyPI metadata messiness (local version labels,
  micro-pinned/`in`-style python markers, stale Requires-Python vs. wheel
  tags) that reroll correctly declines to convert.